In [1]:
import torch
import torch.nn as nn
from datasets import load_dataset
from torch.nn.functional import scaled_dot_product_attention
import math
import time
import os
from torch.cuda.amp import GradScaler
from torch.nn.parallel import DistributedDataParallel as DDP
import torch.distributed as dist
from transformers import GPT2Tokenizer
from torch.utils.data import DataLoader

class LlamaRMSNorm(nn.Module):
    def __init__(self, d_model, eps=1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d_model))
        self.eps = eps

    def forward(self, x): # return x / torch.sqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight
        norm = x.pow(2).mean(-1, keepdim=True).add(self.eps).sqrt()
        return self.weight * x / norm
    
class LlamaRotaryEmbedding(nn.Module):
    def __init__(self, dim, base=10000.0):
        super().__init__()
        self.dim = dim
        self.base = base
        # Create inv_freq for half the dimension
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq)

    def forward(self, seq_len, device):
        t = torch.arange(seq_len, device=device, dtype=torch.float32)
        sinusoid_inp = torch.einsum('i,j->ij', t, self.inv_freq)  # Outer product
        sin = sinusoid_inp.sin()
        cos = sinusoid_inp.cos()
        return sin, cos

    def apply_rotary_emb(self, x, emb):
        sin, cos = emb
        
        # Reshape sin and cos to match input dimensions
        # Add batch and head dimensions, and ensure the last dimension matches
        sin = sin.view(1, sin.shape[0], 1, sin.shape[1])  # [1, seq_len, 1, dim//2]
        cos = cos.view(1, cos.shape[0], 1, cos.shape[1])  # [1, seq_len, 1, dim//2]
        
        # Split input into half along last dimension
        x_half1, x_half2 = x[..., :x.shape[-1]//2], x[..., x.shape[-1]//2:]
        
        # Apply rotary embeddings
        x_out1 = x_half1 * cos - x_half2 * sin
        x_out2 = x_half2 * cos + x_half1 * sin
        
        # Concatenate back along last dimension
        return torch.cat([x_out1, x_out2], dim=-1)
        
class LlamaSdpaAttention(nn.Module):
    def __init__(self, hidden_size, num_heads):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads

        self.q_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.k_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.v_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.o_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.rotary_emb = LlamaRotaryEmbedding(self.head_dim)

    def forward(self, x, attention_mask=None):
        batch_size, seq_len, _ = x.shape

        # Project queries, keys and values
        q = self.q_proj(x).view(batch_size, seq_len, self.num_heads, self.head_dim)
        k = self.k_proj(x).view(batch_size, seq_len, self.num_heads, self.head_dim)
        v = self.v_proj(x).view(batch_size, seq_len, self.num_heads, self.head_dim)

        # Apply rotary enbeddings
        rotary_emb = self.rotary_emb(seq_len, x.device)
        q = self.rotary_emb.apply_rotary_emb(q, rotary_emb)
        k = self.rotary_emb.apply_rotary_emb(k, rotary_emb)

        # Reshape for attention
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        # Scaled dot product attention
        attn_output = scaled_dot_product_attention(
            q, k, v,
            attn_mask=attention_mask,
            dropout_p=0.0,
            is_causal=True
        )

        # Reshape and project output
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.hidden_size)
        return self.o_proj(attn_output)

class LlamaMLP(nn.Module):
    def __init__(self, hidden_size, intermediate_size):
        super().__init__()
        self.gate_proj = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.up_proj = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.down_proj = nn.Linear(intermediate_size, hidden_size, bias=False)
        self.act_fn = nn.SiLU()

    def forward(self, x):
        return self.down_proj(self.act_fn(self.gate_proj(x) + self.up_proj(x)))
    
class LlamaDecoderLayer(nn.Module):
    def __init__(self, hidden_size, num_heads, intermediate_size):
        super().__init__()
        self.attn = LlamaSdpaAttention(hidden_size, num_heads)
        self.input_layernorm = LlamaRMSNorm(hidden_size)
        self.post_attention_layernorm = LlamaRMSNorm(hidden_size)
        self.mlp = LlamaMLP(hidden_size, intermediate_size)
    
    def forward(self, x, attention_mask=None):
        # Self-attention
        residual = x
        x = self.attn(self.input_layernorm(x), attention_mask)
        x = x + residual

        # Feedforward
        residual = x
        x = self.mlp(self.post_attention_layernorm(x))
        x = x + residual

        return x
    
class LlamaModel(nn.Module):
    def __init__(self, vocab_size, hidden_size, num_hidden_layers, num_heads, intermediate_size):
        super().__init__()
        self.embed_tokens = nn.Embedding(vocab_size, hidden_size)
        self.layers = nn.ModuleList([
            LlamaDecoderLayer(hidden_size, num_heads, intermediate_size)
            for _ in range(num_hidden_layers)
        ])
        self.norm = LlamaRMSNorm(hidden_size)
    
    def forward(self, input_ids, attention_mask=None):

        x = self.embed_tokens(input_ids)

        for layer in self.layers:
            x = layer(x, attention_mask)

        return self.norm(x)
    
class LlamaForCausalLM(nn.Module):
    def __init__(self, vocab_size, hidden_size, num_hidden_layers, num_heads, intermediate_size, max_seq_len):
        super().__init__()
        self.model = LlamaModel(vocab_size, hidden_size, num_hidden_layers, num_heads, intermediate_size)
        self.lm_head = nn.Linear(hidden_size, vocab_size, bias=False)
        self.lm_head.weight = self.model.embed_tokens.weight # Weight Sharing

    def forward(self, input_ids, attention_mask=None):
        x = self.model(input_ids, attention_mask)
        logits = self.lm_head(x)
        return logits
    
    def resize_token_embeddings(self, new_num_tokens):
        """"""
        # Resize the token embeddings
        old_embeddings = self.model.embed_tokens
        new_emmbeddings = nn.Embedding(new_num_tokens, old_embeddings.embedding_dim)

        # Copy the exisiting embeddings
        num_tokens_to_copy = min(old_embeddings.num_embeddings, new_num_tokens)
        new_emmbeddings.weight.data[:num_tokens_to_copy, :] = old_embeddings.weight.data[:num_tokens_to_copy, :]

        # Replace the old embeddings
        self.model.embed_tokens = new_emmbeddings

        # Update the lm_head to match the new embedding size
        old_lm_head = self.lm_head
        self.lm_head = nn.Linear(old_lm_head.in_features, new_num_tokens, bias=False)

        # Copy the exisiting weights
        num_tokens_to_copy = min(old_lm_head.out_features, new_num_tokens)
        self.lm_head.weight.data[:num_tokens_to_copy, :] = old_lm_head.weight.data[:num_tokens_to_copy, :]

        # Maintain weight sharing
        self.lm_head.weight = self.model.embed_tokens.weight

        return self.model.embed_tokens

In [2]:
import torch
from transformers import GPT2Tokenizer
import os

def load_model_for_inference(model, checkpoint_path, device='cuda'):
    """Load the model weights from checkpoint"""
    if not os.path.exists(checkpoint_path):
        raise FileNotFoundError(f"Checkpoint file not found: {checkpoint_path}")
    
    # Load checkpoint
    checkpoint = torch.load(checkpoint_path, map_location=device)
    
    # Load model state dict
    model.load_state_dict(checkpoint['model_state_dict'])
    model = model.to(device)  # Ensure model is on the correct device
    model.eval()  # Set to evaluation mode
    
    print(f"Checkpoint loaded: {checkpoint_path}")
    print(f"Step: {checkpoint['step']}, Loss: {checkpoint['loss']}")
    
    return model

def generate_text(model, tokenizer, prompt, max_length=100, temperature=0.7, device='cuda'):
    """Generate text based on a prompt"""
    # Ensure model is on the correct device
    model = model.to(device)
    
    # Encode the prompt
    inputs = tokenizer(prompt, return_tensors="pt", padding=True, truncation=True)
    # Move input tensors to the same device as the model
    input_ids = inputs["input_ids"].to(device)
    
    with torch.no_grad():
        # Get the model outputs
        outputs = model(input_ids)
        
        # Get the next token predictions
        next_token_logits = outputs[..., -1, :]
        
        # Apply temperature
        next_token_logits = next_token_logits / temperature
        
        # Sample from the distribution
        probs = torch.nn.functional.softmax(next_token_logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        
        generated_ids = torch.cat([input_ids, next_token], dim=-1)
        
        # Continue generating until max_length or EOS
        for _ in range(max_length - len(generated_ids[0])):
            outputs = model(generated_ids)
            next_token_logits = outputs[..., -1, :] / temperature
            probs = torch.nn.functional.softmax(next_token_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            generated_ids = torch.cat([generated_ids, next_token], dim=-1)
            
            if next_token.item() == tokenizer.eos_token_id:
                break
    
    # Decode the generated text
    generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
    return generated_text

def main():
    # Configuration
    checkpoint_path = "checkpoints/model_step_5050.pt"
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # Initialize tokenizer
    tokenizer = GPT2Tokenizer.from_pretrained("HuggingFaceTB/cosmo2-tokenizer")
    tokenizer.pad_token = tokenizer.eos_token
    model_vocab_size = len(tokenizer)

    # Model configuration
    vocab_size = model_vocab_size
    hidden_size = 576
    num_hidden_layers = 30
    num_heads = 9
    intermediate_size = 1536
    max_seq_len = 2048

    # Initialize model and move to device
    model = LlamaForCausalLM(vocab_size, hidden_size, num_hidden_layers, num_heads, 
                            intermediate_size, max_seq_len)
    model = model.to(device)

    # Load model weights
    print("Loading model from checkpoint...")
    model = load_model_for_inference(model, checkpoint_path, device)

    # Print model size
    model_size = sum(p.numel() for p in model.parameters())
    print(f"Model Size: {model_size / 1e6:.2f}M parameters")
    
    # Example prompts for testing
    test_prompts = [
        "Once upon a time",
        "The future of artificial intelligence",
        "In the distant galaxy"
    ]
    
    # Generate text for each prompt
    print("\nGenerating text...\n")
    for prompt in test_prompts:
        print(f"Prompt: {prompt}")
        generated_text = generate_text(
            model,
            tokenizer,
            prompt,
            max_length=100,
            temperature=0.7,
            device=device
        )
        print(f"Generated text: {generated_text}\n")

if __name__ == "__main__":
    main()

Loading model from checkpoint...


/tmp/ipykernel_248797/2390165423.py:11: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path, map_location=device)
Asking to truncate to max

Checkpoint loaded: checkpoints/model_step_5050.pt
Step: 5050, Loss: 0.48327672481536865
Model Size: 147.79M parameters

Generating text...

Prompt: Once upon a time
Generated text: Once upon a time, in a beautiful forest, there lived many different creatures who worked together to grow and thrive. Each one had its own unique way of caring for one another and their habitats. This made a big difference in how humans hugging and play with each other.

One day, a new friend named Max detectives decided to build a two-year-old guy named Max teary and his best friend, Max. They started by checking the restaurant and asked why it was important.

Prompt: The future of artificial intelligence
Generated text: The future of artificial intelligence (AI) and its impact on human behavior. Despite its significance in modern society, this traumatic event has left indelible marks on human experience, particularly in the context of the human condition. In this section, we will explore the concept of neu